# Computational Theory Assignment Problem (SHA-256)


## Problem 1: Representing SHA-256 Data
Before we go ahead with the implementation of SHA-256, I'll first clarify a few things, mainly how input, output, and intermediate data in Python.

NOTE: the small python example I give for each data representation is just a simplified example for demo's sake, the actual implementation is after that.

### 32-bit words
Unlike C/C++/Rust where there's explicit `uint32_t` typing that I would love to use, python is dynamically typed and the way its integer behaves is not the same, it is not fixed-width, which in sha-256 implementation's case, is exactly the type of behaviour I would need to preserve. Initially, I wanted to go with a `Int32` class wrapper to make it safer, but decided against it because of a few reasons.
1. Performance overhead - sha256 produces a 32-bit result in every single operation, with a class, in the thousands of loops that gets ran in the function there would be a LOT of object allocation overhead
2. Lose out on clean bitwise op syntax - this one is especially painful because python doesn't even enforce static type bounds on a custom class, meaning the `Int32` class is just wrapping a magically expanding python int under the hood anyway, so every single math op still requires me to manually slice it back down to 32 bits. I would need to overload so many operators (i.e. `__add__`,`__xor__`, `__rshift__`, etc. you get the idea), which adds a lot of work for no reason, not to mention more places where things could go wrong. 
3. KISS principle, this one is self explanatory

So, in the end, I'll just go ahead with the native python int with a 32bit mask (`& 0xFFFFFFFF`), nice and simple, if it overflows, the overflowed bits would get masked off anyway and to make things smoother and easier to read, we can add a bunch of helper functions to keep things nice and clean.

In [ ]:
# this is just for demo, we'll be encapsulating the masking in a helper function later on
word = 0x12345678
word = word & 0xffffffff

print("word in decimal = ", word) # 305419896 (0x12345678 in decimal)
print("word as bytes = ", f"{word:08x}")  # 12345678
# x means hex, 08 means 8 digit, pad if necessary, same format as in C printf

word in decimal =  305419896
word as bytes =  12345678


### sequences of 32-bit words
I'll be using `list[int]` to represent the sequence of words, since we're already using the native ints anyways, a typical python list should be a no brainer, it is random access, O(1) time, easy to initialize. It makes it clearer if I explain why we're NOT using the alternatives.
1. python array module - it does have a native uint 32 type, but that adds unnecessary type conversion when I put the words in or access them, once again, the KISS principle
2. tuple - tuples are immutable, we need to mutate them during a step in sha256, straightforward
3. bytes/bytearray - This is an alternative for both the 32bit word and sequence of the words, and I suppose it would work fine BUT I would need to call `int.from_bytes()` every time I want to perform addition or bitwise op on a word, this is really unnecessary.

In [10]:
words = [0x12345678, 0x90abcdef, 0x00000001] # list of 3 words as a demo

print("len(words) = ", len(words))
print([f"{word:08x}" for word in words])

len(words) =  3
['12345678', '90abcdef', '00000001']


### input messages
For the input messages we'll use python's immutable `bytes` type because sha256 operates at a byte level, the `bytes` type is evidently byte-level precise, I'm not going to be using characters because UTF-8 strings can range from 1-4 bytes long e.g. ඞ (0xE0 0xB6 0x9E) is 3 bytes long meanwhile the char `a` is 1 byte long. 

To give further clarification on python's `bytes` type, under the hood, it's literally just a contiguous array of 1 byte values, but with python magic wrapped over it. TLDR it behaves like `const uint8_t[]`. It's also handy because out of the box it provides O(1) slicing that's easy to use. An example of the slicing behaviour is in the code below.

I suppose we could theoretically use `str` or `list` here too but that would require extra work. 

e.g. using `str` forces me to encode to bytes before calculating exact bit lengths or applying masks anyway

e.g.2 using `list[int]` of raw byte values wastes memory, loses the super easy byte slicing provided by `bytes` and also require extra conversion steps during the block processing, which is really unnecessary

In [ ]:
message = "abc"
message_bytes = message.encode("utf-8")

print("message_bytes = ", message_bytes)  # b'abc', 
# it should actually be the byte val but python tries to be helpful and shows the ascii representation of the bytes if they are printable
print("len(message_bytes) = ", len(message_bytes))  # 3

# slicing example
block = b"\x01\x02\x03\x04\x05\x06\x07\x08\x09\x0A" # this would be an immutable bytes object with 10 bytes
# slicing -> returns a 4-byte 'bytes' object
chunk = block[0:4]   # chunk = b'\x01\x02\x03\x04' (type: bytes)
print("chunk = ", chunk)


message_bytes =  b'abc'
len(message_bytes) =  3
chunk =  b'\x01\x02\x03\x04'


### 512-bit message blocks
Once the input message has been padded, sha256 processes it in 512bit chunks (64 bytes) AKA message blocks. Since the input msg is represented in `bytes`, naturally, each individual block would be a subslice of that big padded input message, meaning they would all be bytes that are 64-byte wide (let's ignore the metadata python attaches to the `bytes` object). And obviously, if there's more than 1 block, I'll be using a standard list ie `list[bytes]`

In [ ]:
# initializing an individual block just for demo
block = bytes(range(64))

assert len(block) == 64
assert len(block) * 8 == 512

# in practice it'll be a direct consq of the padded message, it'll look something like this, which we'll impl again later
# I'll explain the padding later too, just know that it's always a multiple of 64 bytes
padded_msg = b"abc" + b"\x80" + b"\x00" * 60 + (24).to_bytes(8, "big") 
blocks = [
    padded_msg[i : i + 64] 
    for i in range(0, len(padded_msg), 64)
]

### the final 256-bit hash value
There's actually 2 useful representation for this, the final hash can be:
1. a 32 bytes for binary ops or storage
2. 64-char hexadecimal string for display (people usually display hashes in this because if we actually printed the 32 raw bytes, it'll be unreadable gibberish since many byte values won't correspond to printable ascii characters)

In [18]:
digest_bytes = bytes.fromhex(
    "ba7816bf8f01cfea414140de5dae2223"
    "b00361a396177a9cb410ff61f20015ad"
)

assert len(digest_bytes) == 32
assert len(digest_bytes) * 8 == 256

print()
print("the raw bytes = ", digest_bytes)  # b'\xba\x78\x16\xbf... you get the idea, but remember the python magic i talked about above, so it might show some ascii when printed
digest_hex = digest_bytes.hex()
print("the hex display = ", digest_hex)  # ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


the raw bytes =  b'\xbax\x16\xbf\x8f\x01\xcf\xeaAA@\xde]\xae"#\xb0\x03a\xa3\x96\x17z\x9c\xb4\x10\xffa\xf2\x00\x15\xad'
the hex display =  ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad


### big-endian integers
big-endian just means it stores the most significant byte first, e.g. the integer 0x12345678 is stored as:
```
12 34 56 78
```
The endianness only really matters at 2 places for sha256:
1. at the input boundary: when we split the 64 byte message block (512bit) into 16 intial 32-bit words ($W_0 \dots W_{15}$). The FIPS standard says to interprete each 4byte group as big endian 32bit int, so that's what we're following
2. at the output: when we convert the final 32bit state variables ($H_0 \dots H_7$) back into the final 32-byte hash array, once again just following FIPS, we write out each state integer as 4 big-endian bytes.

I'll show the 2 in out boundaries in the code below

NOTE: 
inside the main loop, during the math ops (bit shifts, rotations, add32, Ch, Maj) we'll just be working with native python ints as stated above, we won't be messing with endianness in there, not that the ints themselves have any inherent endianness in the first place, because endianness only applies when bytes are in a sequence (in memory, in a file or in a stream)

In [19]:
value = 0x12345678

encoded = value.to_bytes(4, byteorder="big")
decoded = int.from_bytes(encoded, byteorder="big")

print(encoded)  # b'\x124Vx'
print(decoded == value)  # True

b'\x124Vx'
True
